In [15]:
# Loads up the imports

from pathlib import Path
import random

import pandas as pd

import numpy as np
from PIL import Image
from skimage.feature import local_binary_pattern

In [16]:
# Sets up the constants

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "training"
FEATURE_DIR = Path("outputs") / "features"
CLASSES = ["0", "1", "2", "3"]
VALIDATION_FRACTION = 0.20
SEED = 2100
LBP_POINTS = 8
LBP_RADIUS = 1
VGG16_IMAGE_SIZE = (224, 224)
VGG16_BATCH_SIZE = 16

In [17]:
# Load images and create a DataFrame with their paths, classes, and patient IDs.

image_records = []

for class_label in CLASSES:
    class_folder = TRAIN_DIR / class_label
    image_paths = sorted(class_folder.glob("*.png"))

    for image_path in image_paths:
        image_records.append(
            {
                "path": image_path,
                "label": int(class_label),
                "patient": image_path.name.split("_", maxsplit=1)[0],
            }
        )

training_data = pd.DataFrame(image_records, columns=["path", "label", "patient"])

if training_data.empty:
    raise FileNotFoundError(
        f"Nenhuma imagem PNG encontrada em {TRAIN_DIR.resolve()}. "
        f"Pasta atual: {Path.cwd()}. Confirme que data/training/<classe>/ existe."
    )

print(f"Imagens carregadas: {len(training_data)}")
print(f"Pacientes na base: {training_data['patient'].nunique()}")

Imagens carregadas: 593
Pacientes na base: 20


In [18]:
# Split each class by patient so images from one patient cannot enter both sets.
# Select 20% of that class's patients for validation; with five patients, this selects one.
# A fixed seed makes the random patient selection reproducible.

random_generator = random.Random(SEED)
validation_patients = set()

for class_label, class_data in training_data.groupby("label", sort=True):
    class_patients = sorted(class_data["patient"].unique())
    validation_count = round(len(class_patients) * VALIDATION_FRACTION)
    selected_validation_patients = random_generator.sample(class_patients, validation_count)
    validation_patients.update(selected_validation_patients)
    print(
        f"Classe {class_label}: pacientes selecionados para validação: "
        f"{selected_validation_patients}"
    )

# Put all images from selected patients in validation; keep all remaining patients in training.

validation_mask = training_data["patient"].isin(validation_patients)
validation_data = training_data.loc[validation_mask].copy().reset_index(drop=True)
train_data = training_data.loc[~validation_mask].copy().reset_index(drop=True)

# Show how many images were assigned to each partition.

print(f"Pacientes de validação: {sorted(validation_patients)}")
print(f"Imagens para treino: {len(train_data)}")
print(f"Imagens para validação: {len(validation_data)}")

Classe 0: pacientes selecionados para validação: ['18']
Classe 1: pacientes selecionados para validação: ['24']
Classe 2: pacientes selecionados para validação: ['04']
Classe 3: pacientes selecionados para validação: ['19']
Pacientes de validação: ['04', '18', '19', '24']
Imagens para treino: 475
Imagens para validação: 118


In [19]:
# Combine both partitions so they can be summarized in one table.

all_data = pd.concat(
    [
        train_data.assign(split="Treino"),
        validation_data.assign(split="Validação"),
    ],
    ignore_index=True,
)

# Count images and distinct patients for each partition and class.

split_summary = (
    all_data.groupby(["split", "label"])
    .agg(images=("path", "size"), patients=("patient", "nunique"))
)

# Display the counts to check the resulting split.

split_summary

images  patients
split     label                  
Treino    0         118         4
          1         117         4
          2         120         4
          3         120         4
Validação 0          28         1
          1          30         1
          2          30         1
          3          30         1

In [20]:
# Compute LBP on grayscale images and use a normalized histogram as each feature vector.

FEATURE_DIR.mkdir(parents=True, exist_ok=True)

lbp_feature_rows = []
histogram_bins = np.arange(0, LBP_POINTS + 3)

for image_path in training_data["path"]:
    grayscale_image = np.asarray(Image.open(image_path).convert("L"))
    lbp_image = local_binary_pattern(
        grayscale_image,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method="uniform",
    )
    histogram, _ = np.histogram(lbp_image.ravel(), bins=histogram_bins)
    lbp_feature_rows.append(histogram / histogram.sum())

lbp_feature_names = [f"lbp_{bin_index:02d}" for bin_index in range(LBP_POINTS + 2)]
lbp_features = pd.DataFrame(lbp_feature_rows, columns=lbp_feature_names)
lbp_csv = pd.concat(
    [training_data[["path", "label", "patient"]].reset_index(drop=True), lbp_features],
    axis=1,
)
lbp_csv["path"] = lbp_csv["path"].astype(str)

lbp_output_path = FEATURE_DIR / "lbp_features.csv"
lbp_csv.to_csv(lbp_output_path, index=False)

print(f"CSV LBP salvo em: {lbp_output_path.resolve()}")
print(f"Formato do CSV LBP: {lbp_csv.shape}")

CSV LBP salvo em: D:\Users\FKZ\Documents\GitHub\breast-cancer-image-classification\outputs\features\lbp_features.csv
Formato do CSV LBP: (593, 13)


In [21]:
# Use ImageNet-pretrained VGG16 and extract the 4096 values from its penultimate (fc2) layer.
# The first run downloads the pretrained ImageNet weights.

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model

vgg16_classifier = VGG16(weights="imagenet", include_top=True)
vgg16_feature_model = Model(
    inputs=vgg16_classifier.input,
    outputs=vgg16_classifier.layers[-2].output,
)

# Rebuild image metadata from disk so this cell does not depend on a stale training_data variable.
vgg16_records = []
for class_label in CLASSES:
    class_folder = TRAIN_DIR / class_label
    for image_path in sorted(class_folder.glob("*.png")):
        vgg16_records.append(
            {
                "path": image_path,
                "label": int(class_label),
                "patient": image_path.name.split("_", maxsplit=1)[0],
            }
        )
vgg16_data = pd.DataFrame(vgg16_records)

vgg16_feature_rows = []

for batch_start in range(0, len(vgg16_data), VGG16_BATCH_SIZE):
    batch_data = vgg16_data.iloc[batch_start : batch_start + VGG16_BATCH_SIZE]
    image_batch = np.stack(
        [
            np.asarray(
                Image.open(image_path)
                .convert("RGB")
                .resize(VGG16_IMAGE_SIZE, Image.Resampling.LANCZOS),
                dtype=np.float32,
            )
            for image_path in batch_data["path"]
        ]
    )
    image_batch = preprocess_input(image_batch)
    batch_features = vgg16_feature_model.predict(image_batch, verbose=0)
    vgg16_feature_rows.extend(batch_features)
    print(f"Imagens processadas pela VGG16: {len(vgg16_feature_rows)}/{len(vgg16_data)}")

vgg16_feature_names = [f"vgg16_fc2_{feature_index:04d}" for feature_index in range(4096)]
vgg16_features = pd.DataFrame(vgg16_feature_rows, columns=vgg16_feature_names)
vgg16_csv = pd.concat(
    [vgg16_data[["path", "label", "patient"]].reset_index(drop=True), vgg16_features],
    axis=1,
)
vgg16_csv["path"] = vgg16_csv["path"].astype(str)

vgg16_output_path = FEATURE_DIR / "vgg16_features.csv"
vgg16_csv.to_csv(vgg16_output_path, index=False)

print(f"CSV VGG16 salvo em: {vgg16_output_path.resolve()}")
print(f"Formato do CSV VGG16: {vgg16_csv.shape}")

Imagens processadas pela VGG16: 16/593
Imagens processadas pela VGG16: 32/593
Imagens processadas pela VGG16: 48/593
Imagens processadas pela VGG16: 64/593
Imagens processadas pela VGG16: 80/593
Imagens processadas pela VGG16: 96/593
Imagens processadas pela VGG16: 112/593
Imagens processadas pela VGG16: 128/593
Imagens processadas pela VGG16: 144/593
Imagens processadas pela VGG16: 160/593
Imagens processadas pela VGG16: 176/593
Imagens processadas pela VGG16: 192/593
Imagens processadas pela VGG16: 208/593
Imagens processadas pela VGG16: 224/593
Imagens processadas pela VGG16: 240/593
Imagens processadas pela VGG16: 256/593
Imagens processadas pela VGG16: 272/593
Imagens processadas pela VGG16: 288/593
Imagens processadas pela VGG16: 304/593
Imagens processadas pela VGG16: 320/593
Imagens processadas pela VGG16: 336/593
Imagens processadas pela VGG16: 352/593
Imagens processadas pela VGG16: 368/593
Imagens processadas pela VGG16: 384/593
Imagens processadas pela VGG16: 400/593
Imagen

In [22]:
# Train each classifier on every row in the feature CSV, as required for this task.
# Scaling is applied to SVM and MLP, but not to Random Forest.

from pathlib import Path

import numpy as np
import pandas as pd

from joblib import dump
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

FEATURE_DIR = Path("outputs") / "features"
MODEL_DIR = Path("outputs") / "models"
MODEL_RANDOM_STATE = 2100
MODEL_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_CSV_PATHS = {
    "lbp": FEATURE_DIR / "lbp_features.csv",
    "vgg16": FEATURE_DIR / "vgg16_features.csv",
}
FEATURE_COLUMN_PREFIXES = {
    "lbp": "lbp_",
    "vgg16": "vgg16_fc2_",
}

trained_models = {}
training_results = []

for extractor_name, feature_csv_path in FEATURE_CSV_PATHS.items():
    feature_data = pd.read_csv(feature_csv_path)
    feature_prefix = FEATURE_COLUMN_PREFIXES[extractor_name]
    feature_columns = [
        column_name
        for column_name in feature_data.columns
        if column_name.startswith(feature_prefix)
    ]
    features = feature_data[feature_columns].to_numpy(dtype=np.float32)
    labels = feature_data["label"].to_numpy(dtype=np.int64)

    classifier_candidates = {
        "random_forest": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
        "svm": make_pipeline(
            StandardScaler(),
            SVC(kernel="rbf", class_weight="balanced"),
        ),
        "mlp": make_pipeline(
            StandardScaler(),
            MLPClassifier(
                hidden_layer_sizes=(64,),
                max_iter=300,
                random_state=MODEL_RANDOM_STATE,
            ),
        ),
    }

    for classifier_name, classifier in classifier_candidates.items():
        classifier.fit(features, labels)
        model_key = f"{extractor_name}_{classifier_name}"
        model_path = MODEL_DIR / f"{model_key}.joblib"

        dump(
            {
                "model": classifier,
                "feature_columns": feature_columns,
                "extractor": extractor_name,
                "classifier": classifier_name,
            },
            model_path,
        )
        trained_models[model_key] = classifier
        training_results.append(
            {
                "extractor": extractor_name,
                "classifier": classifier_name,
                "training_images": len(labels),
                "feature_count": len(feature_columns),
                "training_accuracy": accuracy_score(labels, classifier.predict(features)),
                "model_path": str(model_path),
            }
        )
        print(f"Treinado e salvo: {model_path}")

# Training accuracy is measured on the same images used for fitting; it is not test performance.
training_results = pd.DataFrame(training_results)
training_results

Treinado e salvo: outputs\models\lbp_random_forest.joblib
Treinado e salvo: outputs\models\lbp_svm.joblib


c:\Users\FKZ\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (300) reached and the optimization hasn't converged yet.
  warnings.warn(


Treinado e salvo: outputs\models\lbp_mlp.joblib
Treinado e salvo: outputs\models\vgg16_random_forest.joblib
Treinado e salvo: outputs\models\vgg16_svm.joblib
Treinado e salvo: outputs\models\vgg16_mlp.joblib


,extractor,classifier,training_images,feature_count,training_accuracy,model_path
0,lbp,random_forest,593,10,1.000000,outputs\models\lbp_random_forest.joblib
1,lbp,svm,593,10,0.833052,outputs\models\lbp_svm.joblib
2,lbp,mlp,593,10,0.865093,outputs\models\lbp_mlp.joblib
3,vgg16,random_forest,593,4096,1.000000,outputs\models\vgg16_random_forest.joblib
4,vgg16,svm,593,4096,0.998314,outputs\models\vgg16_svm.joblib
5,vgg16,mlp,593,4096,1.000000,outputs\models\vgg16_mlp.joblib


In [23]:
# Load test image paths, class labels, and patient IDs from the class folders.

TEST_DIR = DATA_DIR / "testing"
test_records = []

for class_label in CLASSES:
    class_folder = TEST_DIR / class_label
    for image_path in sorted(class_folder.glob("*.png")):
        test_records.append(
            {
                "path": image_path,
                "label": int(class_label),
                "patient": image_path.name.split("_", maxsplit=1)[0],
            }
        )

test_data = pd.DataFrame(test_records, columns=["path", "label", "patient"])
if test_data.empty:
    raise FileNotFoundError(
        f"Nenhuma imagem PNG encontrada em {TEST_DIR.resolve()}. "
        f"Pasta atual: {Path.cwd()}. Confirme que data/testing/<classe>/ existe."
    )
print(f"Imagens de teste carregadas: {len(test_data)}")
print(f"Pacientes na base de teste: {test_data['patient'].nunique()}")
test_data.groupby("label").size().rename("images")

Imagens de teste carregadas: 371
Pacientes na base de teste: 13


label
0    101
1     90
2     90
3     90
Name: images, dtype: int64

In [24]:
# Use the same grayscale conversion, uniform LBP settings, and normalized histogram as for training.

test_lbp_feature_rows = []
histogram_bins = np.arange(0, LBP_POINTS + 3)

for image_path in test_data["path"]:
    grayscale_image = np.asarray(Image.open(image_path).convert("L"))
    lbp_image = local_binary_pattern(
        grayscale_image,
        P=LBP_POINTS,
        R=LBP_RADIUS,
        method="uniform",
    )
    histogram, _ = np.histogram(lbp_image.ravel(), bins=histogram_bins)
    test_lbp_feature_rows.append(histogram / histogram.sum())

test_lbp_feature_names = [f"lbp_{bin_index:02d}" for bin_index in range(LBP_POINTS + 2)]
test_lbp_features = pd.DataFrame(test_lbp_feature_rows, columns=test_lbp_feature_names)
test_lbp_csv = pd.concat(
    [test_data.reset_index(drop=True), test_lbp_features],
    axis=1,
)
test_lbp_csv["path"] = test_lbp_csv["path"].astype(str)
test_lbp_output_path = FEATURE_DIR / "test_lbp_features.csv"
test_lbp_csv.to_csv(test_lbp_output_path, index=False)

print(f"Test LBP CSV saved to: {test_lbp_output_path.resolve()}")
print(f"Test LBP CSV shape: {test_lbp_csv.shape}")

Test LBP CSV saved to: D:\Users\FKZ\Documents\GitHub\breast-cancer-image-classification\outputs\features\test_lbp_features.csv
Test LBP CSV shape: (371, 13)


In [25]:
# Extract the same 4096 fc2 features using ImageNet-pretrained VGG16 and identical preprocessing.

from tensorflow.keras.applications import VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input
from tensorflow.keras.models import Model

test_vgg16_classifier = VGG16(weights="imagenet", include_top=True)
test_vgg16_feature_model = Model(
    inputs=test_vgg16_classifier.input,
    outputs=test_vgg16_classifier.layers[-2].output,
)

test_vgg16_feature_rows = []
for batch_start in range(0, len(test_data), VGG16_BATCH_SIZE):
    batch_data = test_data.iloc[batch_start : batch_start + VGG16_BATCH_SIZE]
    image_batch = np.stack(
        [
            np.asarray(
                Image.open(image_path)
                .convert("RGB")
                .resize(VGG16_IMAGE_SIZE, Image.Resampling.LANCZOS),
                dtype=np.float32,
            )
            for image_path in batch_data["path"]
        ]
    )
    image_batch = preprocess_input(image_batch)
    batch_features = test_vgg16_feature_model.predict(image_batch, verbose=0)
    test_vgg16_feature_rows.extend(batch_features)
    print(f"Test images processed by VGG16: {len(test_vgg16_feature_rows)}/{len(test_data)}")

test_vgg16_feature_names = [f"vgg16_fc2_{feature_index:04d}" for feature_index in range(4096)]
test_vgg16_features = pd.DataFrame(test_vgg16_feature_rows, columns=test_vgg16_feature_names)
test_vgg16_csv = pd.concat(
    [test_data.reset_index(drop=True), test_vgg16_features],
    axis=1,
)
test_vgg16_csv["path"] = test_vgg16_csv["path"].astype(str)
test_vgg16_output_path = FEATURE_DIR / "test_vgg16_features.csv"
test_vgg16_csv.to_csv(test_vgg16_output_path, index=False)

print(f"Test VGG16 CSV saved to: {test_vgg16_output_path.resolve()}")
print(f"Test VGG16 CSV shape: {test_vgg16_csv.shape}")

Test images processed by VGG16: 16/371
Test images processed by VGG16: 32/371
Test images processed by VGG16: 48/371
Test images processed by VGG16: 64/371
Test images processed by VGG16: 80/371
Test images processed by VGG16: 96/371
Test images processed by VGG16: 112/371
Test images processed by VGG16: 128/371
Test images processed by VGG16: 144/371
Test images processed by VGG16: 160/371
Test images processed by VGG16: 176/371
Test images processed by VGG16: 192/371
Test images processed by VGG16: 208/371
Test images processed by VGG16: 224/371
Test images processed by VGG16: 240/371
Test images processed by VGG16: 256/371
Test images processed by VGG16: 272/371
Test images processed by VGG16: 288/371
Test images processed by VGG16: 304/371
Test images processed by VGG16: 320/371
Test images processed by VGG16: 336/371
Test images processed by VGG16: 352/371
Test images processed by VGG16: 368/371
Test images processed by VGG16: 371/371
Test VGG16 CSV saved to: D:\Users\FKZ\Document

In [26]:
# Load each saved model and select test feature columns in the exact training order.

from joblib import load

MODEL_DIR = Path("outputs") / "models"
test_feature_csv_paths = {
    "lbp": FEATURE_DIR / "test_lbp_features.csv",
    "vgg16": FEATURE_DIR / "test_vgg16_features.csv",
}

test_prediction_tables = []
for extractor_name, test_feature_csv_path in test_feature_csv_paths.items():
    test_feature_data = pd.read_csv(test_feature_csv_path)

    for classifier_name in ("random_forest", "svm", "mlp"):
        model_key = f"{extractor_name}_{classifier_name}"
        model_artifact = load(MODEL_DIR / f"{model_key}.joblib")
        model_features = test_feature_data[model_artifact["feature_columns"]].to_numpy(dtype=np.float32)
        predicted_labels = model_artifact["model"].predict(model_features)

        prediction_table = test_feature_data[["path", "patient", "label"]].copy()
        prediction_table["extractor"] = extractor_name
        prediction_table["classifier"] = classifier_name
        prediction_table["prediction"] = predicted_labels
        test_prediction_tables.append(prediction_table)
        print(f"Applied {model_key} to {len(predicted_labels)} test images")

test_predictions = pd.concat(test_prediction_tables, ignore_index=True)
prediction_dir = Path("outputs") / "predictions"
prediction_dir.mkdir(parents=True, exist_ok=True)
test_predictions.to_csv(prediction_dir / "test_predictions.csv", index=False)
test_predictions.head()

Applied lbp_random_forest to 371 test images
Applied lbp_svm to 371 test images
Applied lbp_mlp to 371 test images
Applied vgg16_random_forest to 371 test images
Applied vgg16_svm to 371 test images
Applied vgg16_mlp to 371 test images


,path,patient,label,extractor,classifier,prediction
0,data\testing\0\66_HER2_10094.png,66,0,lbp,random_forest,2
1,data\testing\0\66_HER2_10295.png,66,0,lbp,random_forest,2
2,data\testing\0\66_HER2_10497.png,66,0,lbp,random_forest,0
3,data\testing\0\66_HER2_10911.png,66,0,lbp,random_forest,1
4,data\testing\0\66_HER2_11524.png,66,0,lbp,random_forest,2


In [27]:
# Compute one-vs-rest sensitivity, specificity, and F1 for every class and classifier.

from sklearn.metrics import confusion_matrix, f1_score

CLASS_LABELS = [0, 1, 2, 3]
confusion_matrices = {}
class_metric_rows = []
model_metric_rows = []
confusion_matrix_rows = []

for (extractor_name, classifier_name), model_data in test_predictions.groupby(
    ["extractor", "classifier"], sort=True
):
    model_key = f"{extractor_name}_{classifier_name}"
    true_labels = model_data["label"].to_numpy()
    predicted_labels = model_data["prediction"].to_numpy()
    matrix = confusion_matrix(true_labels, predicted_labels, labels=CLASS_LABELS)
    confusion_matrices[model_key] = matrix
    class_f1_scores = f1_score(
        true_labels,
        predicted_labels,
        labels=CLASS_LABELS,
        average=None,
        zero_division=0,
    )

    for class_index, class_label in enumerate(CLASS_LABELS):
        true_positive = matrix[class_index, class_index]
        false_negative = matrix[class_index, :].sum() - true_positive
        false_positive = matrix[:, class_index].sum() - true_positive
        true_negative = matrix.sum() - true_positive - false_negative - false_positive
        sensitivity = true_positive / (true_positive + false_negative)
        specificity = true_negative / (true_negative + false_positive)

        class_metric_rows.append(
            {
                "extractor": extractor_name,
                "classifier": classifier_name,
                "class_label": class_label,
                "sensitivity": sensitivity,
                "specificity": specificity,
                "f1_score": class_f1_scores[class_index],
                "support": int(matrix[class_index, :].sum()),
            }
        )

        for predicted_class_index, predicted_class_label in enumerate(CLASS_LABELS):
            confusion_matrix_rows.append(
                {
                    "extractor": extractor_name,
                    "classifier": classifier_name,
                    "true_label": class_label,
                    "predicted_label": predicted_class_label,
                    "count": int(matrix[class_index, predicted_class_index]),
                }
            )

        print(f"Confusion matrix — {model_key} (rows=true, columns=predicted):")
        print(pd.DataFrame(matrix, index=CLASS_LABELS, columns=CLASS_LABELS))

    model_class_metrics = [
        row for row in class_metric_rows
        if row["extractor"] == extractor_name and row["classifier"] == classifier_name
    ]
    model_metric_rows.append(
        {
            "extractor": extractor_name,
            "classifier": classifier_name,
            "accuracy": (true_labels == predicted_labels).mean(),
            "macro_sensitivity": np.mean([row["sensitivity"] for row in model_class_metrics]),
            "macro_specificity": np.mean([row["specificity"] for row in model_class_metrics]),
            "macro_f1": np.mean([row["f1_score"] for row in model_class_metrics]),
        }
    )

class_metrics = pd.DataFrame(class_metric_rows)
model_metrics = pd.DataFrame(model_metric_rows)
confusion_matrix_data = pd.DataFrame(confusion_matrix_rows)

metrics_dir = Path("outputs") / "metrics"
metrics_dir.mkdir(parents=True, exist_ok=True)
class_metrics.to_csv(metrics_dir / "test_class_metrics.csv", index=False)
model_metrics.to_csv(metrics_dir / "test_model_metrics.csv", index=False)
confusion_matrix_data.to_csv(metrics_dir / "test_confusion_matrices.csv", index=False)

class_metrics

Confusion matrix — lbp_mlp (rows=true, columns=predicted):
    0   1   2   3
0  50  24  27   0
1  11  53  26   0
2  16  21  53   0
3   1   0   8  81
Confusion matrix — lbp_mlp (rows=true, columns=predicted):
    0   1   2   3
0  50  24  27   0
1  11  53  26   0
2  16  21  53   0
3   1   0   8  81
Confusion matrix — lbp_mlp (rows=true, columns=predicted):
    0   1   2   3
0  50  24  27   0
1  11  53  26   0
2  16  21  53   0
3   1   0   8  81
Confusion matrix — lbp_mlp (rows=true, columns=predicted):
    0   1   2   3
0  50  24  27   0
1  11  53  26   0
2  16  21  53   0
3   1   0   8  81
Confusion matrix — lbp_random_forest (rows=true, columns=predicted):
    0   1   2   3
0  47  32  22   0
1  19  48  23   0
2  16  29  40   5
3   1   0   7  82
Confusion matrix — lbp_random_forest (rows=true, columns=predicted):
    0   1   2   3
0  47  32  22   0
1  19  48  23   0
2  16  29  40   5
3   1   0   7  82
Confusion matrix — lbp_random_forest (rows=true, columns=predicted):
    0   1   2   3

,extractor,classifier,class_label,sensitivity,specificity,f1_score,support
0,lbp,mlp,0,0.495050,0.896296,0.558659,101
1,lbp,mlp,1,0.588889,0.839858,0.563830,90
2,lbp,mlp,2,0.588889,0.782918,0.519608,90
3,lbp,mlp,3,0.900000,1.000000,0.947368,90
4,lbp,random_forest,0,0.465347,0.866667,0.510870,101
5,lbp,random_forest,1,0.533333,0.782918,0.482412,90
6,lbp,random_forest,2,0.444444,0.814947,0.439560,90
7,lbp,random_forest,3,0.911111,0.982206,0.926554,90
8,lbp,svm,0,0.495050,0.870370,0.537634,101
9,lbp,svm,1,0.511111,0.786477,0.469388,90


In [28]:
# Use macro F1 as the primary selection metric so each of the four classes weighs equally.

best_model_result = model_metrics.sort_values(
    ["macro_f1", "macro_sensitivity"],
    ascending=False,
).iloc[0]

best_model_name = f"{best_model_result['extractor']} + {best_model_result['classifier']}"
print(f"Melhor modelo: {best_model_name}")
print(f"Métrica de seleção: F1 macro = {best_model_result['macro_f1']:.4f}")
print(f"Sensibilidade macro: {best_model_result['macro_sensitivity']:.4f}")
print(f"Especificidade macro: {best_model_result['macro_specificity']:.4f}")

best_model_result

Melhor modelo: vgg16 + random_forest
Métrica de seleção: F1 macro = 0.8100
Sensibilidade macro: 0.8176
Especificidade macro: 0.9408


extractor                    vgg16
classifier           random_forest
accuracy                  0.822102
macro_sensitivity         0.817574
macro_specificity          0.94081
macro_f1                   0.80996
Name: 4, dtype: object